In [1]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os
import sys
sys.path.append(r'C:\Users\rite\Documents\dataset_discovery_llm_implementation\Dataset-search-LLM\dataset_preprocessing')
import preprocessing_utils as utils
import json
from concurrent.futures import ThreadPoolExecutor, as_completed



In [2]:
client = chromadb.PersistentClient(path="/chroma_db")

embedding_function = utils.OpenAIEmbeddingFunction()

collection = client.get_or_create_collection(
    name="test_instructions_ottqa",
    embedding_function=OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small"
    )
)

In [3]:
collection.count()

7260

In [4]:
model = "gpt-5-nano-2025-08-07"

In [5]:
from datasets import load_dataset

ds = load_dataset("target-benchmark/ottqa-queries", split="validation")

In [6]:
df = ds.to_pandas()

In [7]:
queries = df['query'].tolist()

In [8]:
batch_size = 10
batches = [queries[i:i + batch_size] for i in range(0, len(queries), batch_size)]

In [9]:
batches[0]

['Who created the series in which the character of Robert , played by actor Nonso Anozie , appeared ?',
 'What did the 2nd championship win at the Sevens Grand Prix Series for the team with the most top 4 finishes qualify them for ?',
 "This 70 's Kishore Kumar song was in a film produced by Alankar Chitra and directed by Shanker Mukherjee ?",
 'What is the full birth name of the Bradford A.F.C player that only played for the team in 2011 ?',
 'How many academic staff are at the university in Budapest that has the official abbreviation BME ?',
 'The 1995 Tooheys 1000 driver who was second-to-last in the Tooheys Top 10 was born where ?',
 'What is the capacity of the home grounds of the club a player transfered from Arsenal FC to FC Dordecht ?',
 'What is the full name of the person who is a NYU prize winner alumnus associated with ARTS ?',
 'What position does 2009–10 season Vancouver Canucks player Rob Davison currently hold with the Toronto Marlies ?',
 'What is the best known song s

In [10]:
def process_batch(batch_id, batch, top_k):
    print(f"Start batch {batch_id} with {len(batch)} queries")
    results = []

    for query in batch:
        try:
            res = utils.get_results_from_query_different_recall(query, model, collection)
            results.append((query, res))
        except Exception as e:
            print(f"Error in batch {batch_id}, query '{query}': {e}")

    print(f"Batch {batch_id} completed")
    return results

In [11]:
results_all_batches = []
with ThreadPoolExecutor(max_workers=8) as executor:

    futures = {
        executor.submit(process_batch, i + 1, batch, 5): i + 1
        for i, batch in enumerate(batches)
    }

    for future in as_completed(futures):
        batch_id = futures[future]
        try:
            result = future.result()
            results_all_batches.append(result)
        except Exception as e:
            print(f"Error in batch {batch_id}: {e}")

print("All batches completed")

Start batch 1 with 10 queries
Start batch 2 with 10 queries
Start batch 3 with 10 queries
Start batch 4 with 10 queries
Start batch 5 with 10 queries
Start batch 6 with 10 queries
Start batch 7 with 10 queries
Start batch 8 with 10 queries
Batch 7 completed
Start batch 9 with 10 queries
Batch 5 completed
Start batch 10 with 10 queries
Batch 1 completed
Start batch 11 with 10 queries
Batch 4 completed
Start batch 12 with 10 queries
Batch 3 completed
Start batch 13 with 10 queries
Batch 2 completed
Start batch 14 with 10 queries
Batch 8 completed
Start batch 15 with 10 queries
Batch 6 completed
Start batch 16 with 10 queries
Batch 9 completed
Start batch 17 with 10 queries
Batch 11 completed
Start batch 18 with 10 queries
Batch 13 completed
Start batch 19 with 10 queries
Batch 12 completed
Start batch 20 with 10 queries
Batch 14 completed
Start batch 21 with 10 queries
Batch 10 completed
Start batch 22 with 10 queries
Batch 15 completed
Start batch 23 with 10 queries
Batch 16 completed
S

In [12]:
results_all_batches

[[('What is the date of birth of the Canadian football offensive lineman that was drafted tenth overall by the Winnipeg Blue Bombers in the 2016 CFL offseason Draft ?',
   {'1': [('8126', '2016_Winnipeg_Blue_Bombers_season_0')],
    '3': [('8126', '2016_Winnipeg_Blue_Bombers_season_0'),
     ('8727', '2008_Major_League_Baseball_Draft_2'),
     ('8608', '2012_MLS_SuperDraft_1')],
    '5': [('8126', '2016_Winnipeg_Blue_Bombers_season_0'),
     ('8727', '2008_Major_League_Baseball_Draft_2'),
     ('8608', '2012_MLS_SuperDraft_1'),
     ('8386', 'Washington_Redskins_draft_history_88'),
     ('8102', '1979_Detroit_Lions_season_0')],
    '10': [('8126', '2016_Winnipeg_Blue_Bombers_season_0'),
     ('8727', '2008_Major_League_Baseball_Draft_2'),
     ('8608', '2012_MLS_SuperDraft_1'),
     ('8386', 'Washington_Redskins_draft_history_88'),
     ('8102', '1979_Detroit_Lions_season_0'),
     ('8469', '1997_AFL_draft_1'),
     ('8171', '2007_AFL_Draft_4'),
     ('8629', '2011_NFL_Draft_2'),
     

In [13]:
with open("query_optimization_ottqa_result_recall_1_3_5_10.json", "w", encoding="utf-8") as f:
    json.dump(results_all_batches, f, ensure_ascii=False, indent=4)